In [1]:
!pip -q install gradio transformers torch pandas numpy

In [2]:
import numpy as np
import pandas as pd
import torch
import gradio as gr

from google.colab import drive
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

drive.mount("/content/drive")

print("Gradio:", gr.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
Gradio: 6.27.0
GPU available: True
GPU: Tesla T4


In [3]:
MODEL_PATH = (
    "/content/drive/MyDrive/"
    "distilbert_llm_router"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(DEVICE)
model.eval()

print("Model loaded on:", DEVICE)
print("Outputs:", model.config.num_labels)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded on: cuda
Outputs: 11


In [4]:
canonical_df = pd.read_pickle(
    "/content/drive/MyDrive/"
    "router_project_canonical.pkl"
)

splits = np.load(
    "/content/drive/MyDrive/"
    "router_split_indices.npz"
)

FINAL_THRESHOLD = float(
    np.load(
        "/content/drive/MyDrive/"
        "distilbert_final_threshold.npy"
    )[0]
)

distilbert_test_pred = np.load(
    "/content/drive/MyDrive/"
    "distilbert_test_predictions.npy"
)

distilbert_test_choices = np.load(
    "/content/drive/MyDrive/"
    "distilbert_test_choices.npy"
)

train_idx = splits["train_idx"]
test_idx = splits["test_idx"]

print("Frozen threshold:", FINAL_THRESHOLD)
print("Test questions:", len(test_idx))

Frozen threshold: 0.94
Test questions: 2297


In [5]:
MODEL_COLS = [
    "WizardLM/WizardLM-13B-V1.2",
    "claude-instant-v1",
    "claude-v1",
    "claude-v2",
    "gpt-3.5-turbo-1106",
    "gpt-4-1106-preview",
    "meta/code-llama-instruct-34b-chat",
    "meta/llama-2-70b-chat",
    "mistralai/mistral-7b-chat",
    "mistralai/mixtral-8x7b-chat",
    "zero-one-ai/Yi-34B-Chat"
]

DISPLAY_NAMES = [
    "WizardLM-13B",
    "Claude Instant",
    "Claude v1",
    "Claude v2",
    "GPT-3.5 Turbo",
    "GPT-4",
    "CodeLlama-34B",
    "Llama-2-70B",
    "Mistral-7B",
    "Mixtral-8x7B",
    "Yi-34B"
]

GPT4_INDEX = MODEL_COLS.index(
    "gpt-4-1106-preview"
)

In [6]:
X = canonical_df[
    [
        "sample_id",
        "prompt",
        "domain",
        "eval_name"
    ]
].copy()

Y = pd.DataFrame({
    model:
    canonical_df[
        f"{model}|performance"
    ]
    for model in MODEL_COLS
})

C = pd.DataFrame({
    model:
    canonical_df[
        f"{model}|cost"
    ]
    for model in MODEL_COLS
})

X_test = X.iloc[
    test_idx
].reset_index(drop=True)

Y_test = Y.iloc[
    test_idx
].reset_index(drop=True)

C_test = C.iloc[
    test_idx
].reset_index(drop=True)

C_train = C.iloc[
    train_idx
].reset_index(drop=True)

mean_train_cost = (
    C_train.mean().to_numpy()
)

print("Test:", X_test.shape)
print("Models:", len(MODEL_COLS))

Test: (2297, 4)
Models: 11


In [7]:
MAX_LENGTH = 384


def predict_scores(question):

    inputs = tokenizer(
        question,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True
    )

    inputs = {
        key: value.to(DEVICE)
        for key, value
        in inputs.items()
        if key != "token_type_ids"
    }

    with torch.no_grad():

        outputs = model(
            **inputs
        )

    scores = (
        outputs.logits
        .squeeze(0)
        .float()
        .cpu()
        .numpy()
    )

    scores = np.clip(
        scores,
        0.0,
        1.0
    )

    return scores

In [8]:
def select_model(scores):

    cost_order = np.argsort(
        mean_train_cost
    )

    eligible = [
        i
        for i in cost_order
        if scores[i] >= FINAL_THRESHOLD
    ]

    if len(eligible) > 0:

        selected = eligible[0]

        reason = (
            "Cheapest model whose predicted "
            f"performance is ≥ {FINAL_THRESHOLD:.2f}"
        )

    else:

        selected = int(
            np.argmax(scores)
        )

        reason = (
            "No model crossed the threshold, "
            "so the highest predicted performer "
            "was selected."
        )

    return selected, reason

In [9]:
test_question = """
A car travels 120 km in 2 hours.
What is its average speed?
"""

scores = predict_scores(
    test_question
)

selected, reason = select_model(
    scores
)

print(
    "Selected:",
    DISPLAY_NAMES[selected]
)

print(
    "Predicted performance:",
    scores[selected]
)

print(
    "Estimated cost:",
    mean_train_cost[selected]
)

print(
    "Reason:",
    reason
)

Selected: GPT-4
Predicted performance: 0.62890255
Estimated cost: 0.003713676
Reason: No model crossed the threshold, so the highest predicted performer was selected.


In [10]:
def live_router(question):

    if not question.strip():

        return (
            "Please enter a question.",
            pd.DataFrame()
        )

    scores = predict_scores(
        question
    )

    selected, reason = select_model(
        scores
    )

    selected_cost = (
        mean_train_cost[selected]
    )

    gpt4_cost = (
        mean_train_cost[GPT4_INDEX]
    )

    savings = (
        1
        - selected_cost / gpt4_cost
    ) * 100

    eligible = (
        scores >= FINAL_THRESHOLD
    )

    table = pd.DataFrame({
        "Model": DISPLAY_NAMES,
        "Predicted Performance":
            scores.round(4),

        "Estimated Historical Cost ($)":
            mean_train_cost.round(7),

        "Above Threshold":
            eligible
    })

    table["Selected"] = False

    table.loc[
        selected,
        "Selected"
    ] = True

    table = table.sort_values(
        "Predicted Performance",
        ascending=False
    )

    summary = f"""
## Routing Decision

**Selected model:** `{DISPLAY_NAMES[selected]}`

**Predicted performance:** `{scores[selected]:.3f}`

**Frozen threshold:** `{FINAL_THRESHOLD:.2f}`

**Estimated historical cost:** `${selected_cost:.6f}`

**GPT-4 historical average cost:** `${gpt4_cost:.6f}`

**Estimated saving vs GPT-4:** `{savings:.2f}%`

**Reason:** {reason}

> The predicted value is a RouterBench performance score estimate, not a calibrated probability.
"""

    return summary, table

In [11]:
def benchmark_replay(index):

    index = int(index)

    if index < 0:
        index = 0

    if index >= len(X_test):
        index = len(X_test) - 1

    question = X_test.loc[
        index,
        "prompt"
    ]

    domain = X_test.loc[
        index,
        "domain"
    ]

    selected = int(
        distilbert_test_choices[
            index
        ]
    )

    predictions = (
        distilbert_test_pred[
            index
        ]
    )

    actual_router_perf = (
        Y_test.iloc[
            index,
            selected
        ]
    )

    actual_router_cost = (
        C_test.iloc[
            index,
            selected
        ]
    )

    gpt4_perf = (
        Y_test.iloc[
            index,
            GPT4_INDEX
        ]
    )

    gpt4_cost = (
        C_test.iloc[
            index,
            GPT4_INDEX
        ]
    )

    saving = (
        1
        - actual_router_cost
        / gpt4_cost
    ) * 100

    difference = (
        actual_router_perf
        - gpt4_perf
    )

    if difference > 0:

        comparison = (
            "Router-selected model performed "
            "better than GPT-4."
        )

    elif difference < 0:

        comparison = (
            "Router-selected model performed "
            "worse than GPT-4."
        )

    else:

        comparison = (
            "Router-selected model matched "
            "GPT-4's recorded performance."
        )

    summary = f"""
## Benchmark Replay

**Test example:** `{index}`

**Domain:** `{domain}`

**Selected model:** `{DISPLAY_NAMES[selected]}`

**Router predicted score:** `{predictions[selected]:.3f}`

### Recorded RouterBench result

**Selected model performance:** `{actual_router_perf:.3f}`

**GPT-4 performance:** `{gpt4_perf:.3f}`

**Selected model cost:** `${actual_router_cost:.6f}`

**GPT-4 cost:** `${gpt4_cost:.6f}`

**Cost saving:** `{saving:.2f}%`

### Comparison

{comparison}
"""

    table = pd.DataFrame({
        "Model": DISPLAY_NAMES,
        "Predicted Performance":
            predictions.round(4),

        "Actual Performance":
            Y_test.iloc[
                index
            ].to_numpy(),

        "Actual Cost ($)":
            C_test.iloc[
                index
            ].to_numpy()
    })

    table["Selected"] = False

    table.loc[
        selected,
        "Selected"
    ] = True

    table = table.sort_values(
        "Predicted Performance",
        ascending=False
    )

    return (
        str(question),
        summary,
        table
    )

In [17]:
with gr.Blocks(
    title="Cost-Aware LLM Router"
) as demo:

    gr.Markdown(
        """
# Cost-Aware LLM Router

This project uses a fine-tuned **DistilBERT router**
to estimate the expected RouterBench performance of
11 candidate LLMs.

The routing policy selects the cheapest model that
satisfies a chosen performance threshold.

### Final Test Result

- DistilBERT Router Performance: **0.7604**
- GPT-4 Fixed Performance: **0.7681**
- Historical Average Cost Saving: **~23%**
"""
    )

    # ==================================================
    # TAB 1 — LIVE ROUTER
    # ==================================================

    with gr.Tab("Live Router"):

        gr.Markdown(
            """
## Live Question Routing

### Evaluation Mode
Uses the official frozen threshold from our experiment:

**0.94**

### Exploration Mode
Allows the threshold to be changed to demonstrate the
trade-off between model quality and cost.

Exploration Mode does **not** change the reported
experimental results.
"""
        )

        question_input = gr.Textbox(
            label="Enter a question",
            placeholder=(
                "Example: What is the derivative of x^2?"
            ),
            lines=5
        )

        mode_input = gr.Radio(
            choices=[
                "Evaluation Mode",
                "Exploration Mode"
            ],
            value="Evaluation Mode",
            label="Routing Mode"
        )

        threshold_slider = gr.Slider(
            minimum=0.50,
            maximum=0.99,
            value=FINAL_THRESHOLD,
            step=0.01,
            label=(
                "Exploration Threshold "
                "(ignored in Evaluation Mode)"
            )
        )

        route_button = gr.Button(
            "Route Question",
            variant="primary"
        )

        routing_summary = gr.Markdown()

        routing_table = gr.Dataframe(
            label="Predicted Model Performance",
            interactive=False
        )

        # IMPORTANT:
        # This MUST remain inside gr.Blocks()
        route_button.click(
            fn=live_router,
            inputs=[
                question_input,
                mode_input,
                threshold_slider
            ],
            outputs=[
                routing_summary,
                routing_table
            ]
        )


    # ==================================================
    # TAB 2 — BENCHMARK REPLAY
    # ==================================================

    with gr.Tab("Benchmark Replay"):

        gr.Markdown(
            """
## Benchmark Replay

Select a question from the untouched test set.

This mode shows the router's frozen decision together
with the actual historical RouterBench performance and
cost recorded for the selected LLM and GPT-4.
"""
        )

        example_index = gr.Slider(
            minimum=0,
            maximum=len(X_test) - 1,
            value=0,
            step=1,
            label="Test Question Index"
        )

        replay_button = gr.Button(
            "Replay Routing Decision",
            variant="primary"
        )

        replay_question = gr.Textbox(
            label="Benchmark Question",
            lines=8,
            interactive=False
        )

        replay_summary = gr.Markdown()

        replay_table = gr.Dataframe(
            label="Model Comparison",
            interactive=False
        )

        # This also MUST remain inside gr.Blocks()
        replay_button.click(
            fn=benchmark_replay,
            inputs=[
                example_index
            ],
            outputs=[
                replay_question,
                replay_summary,
                replay_table
            ]
        )


    # ==================================================
    # DISCLAIMER
    # ==================================================

    gr.Markdown(
        """
---

### Important Limitation

This application is an **LLM routing demo**, not a chatbot.

The router predicts which historical RouterBench model
would be appropriate for the question.

It does not actually call GPT-4, Claude, Mixtral, or the
other LLMs to generate an answer.

The costs shown are based on historical RouterBench data.
"""
    )


print("Gradio interface created successfully.")

Gradio interface created successfully.


In [19]:
def select_model(scores, threshold):

    cost_order = np.argsort(mean_train_cost)

    eligible = [
        i
        for i in cost_order
        if scores[i] >= threshold
    ]

    if len(eligible) > 0:

        selected = eligible[0]

        reason = (
            f"{len(eligible)} model(s) crossed "
            f"the threshold {threshold:.2f}. "
            "The cheapest eligible model was selected."
        )

    else:

        selected = int(
            np.argmax(scores)
        )

        reason = (
            f"No model crossed threshold {threshold:.2f}. "
            "The router therefore used its fallback rule "
            "and selected the model with the highest "
            "predicted performance."
        )

    return selected, reason, len(eligible)

In [20]:
def live_router(question, mode, demo_threshold):

    if not question.strip():
        return (
            "Please enter a question.",
            pd.DataFrame()
        )

    scores = predict_scores(question)

    # Official experimental mode
    if mode == "Evaluation Mode":
        threshold = FINAL_THRESHOLD

    # Demonstration-only mode
    else:
        threshold = float(demo_threshold)

    selected, reason, eligible_count = select_model(
        scores,
        threshold
    )

    selected_cost = mean_train_cost[selected]
    gpt4_cost = mean_train_cost[GPT4_INDEX]

    savings = (
        1
        - selected_cost / gpt4_cost
    ) * 100

    eligible = scores >= threshold

    table = pd.DataFrame({
        "Model": DISPLAY_NAMES,
        "Predicted Performance":
            scores.round(4),

        "Estimated Historical Cost ($)":
            mean_train_cost.round(7),

        "Above Threshold":
            eligible
    })

    table["Selected"] = False

    table.loc[
        selected,
        "Selected"
    ] = True

    table = table.sort_values(
        "Predicted Performance",
        ascending=False
    )

    if mode == "Evaluation Mode":

        warning = """
**Evaluation Mode:** Uses the frozen threshold from
our final experiment. These settings correspond to
the reported project methodology.
"""

    else:

        warning = """
**Exploration Mode:** The threshold is being changed
for demonstration purposes only. Results from this mode
are NOT part of the reported test experiment.
"""

    summary = f"""
## Routing Decision

**Mode:** `{mode}`

**Selected model:** `{DISPLAY_NAMES[selected]}`

**Predicted performance:** `{scores[selected]:.3f}`

**Threshold used:** `{threshold:.2f}`

**Models above threshold:** `{eligible_count}`

**Estimated historical cost:** `${selected_cost:.6f}`

**GPT-4 historical average cost:** `${gpt4_cost:.6f}`

**Estimated saving vs GPT-4:** `{savings:.2f}%`

### Why was this model selected?

{reason}

{warning}

> Predicted performance is a RouterBench score estimate,
> not a calibrated probability.
"""

    return summary, table

In [18]:
demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7b9306d97a016e604b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7b9306d97a016e604b.gradio.live
